
<style>
.minimal-navbar {
  position: fixed;
  top: 0;
  left: 0;
  width: 100%;
  height: 50px;
  background: white !important;
  border-bottom: 1px solid black !important;
  z-index: 9999 !important;
  display: flex !important;
  align-items: center !important;
  padding: 0 40px !important;
  font-size: 14px !important;
  font-weight: 600 !important;
  text-transform: uppercase !important;
  letter-spacing: 0.05em !important;
  font-family: sans-serif !important;
}
.minimal-navbar a {
  color: black !important;
  text-decoration: none !important;
  margin-right: 24px !important;
}
.minimal-navbar a:hover {
  text-decoration: underline !important;
}
body {
  padding-top: 60px !important;
}
/* Aggressive styles to override Jupyter */
.jp-InputPrompt, .jp-OutputPrompt, div.prompt { display: none !important; }
#notebook, .jp-Notebook, .container, #notebook-container { 
    max-width: 1000px !important; 
    margin: 0 auto !important; 
    padding: 60px 40px !important; 
}
h1, h2, h3 { border-bottom: 1px solid black !important; padding-bottom: 10px !important; margin-top: 40px !important; }
</style>

<div class="minimal-navbar">
    <a href="#">Dashboard</a>
    <a href="#section-1">Regional</a>
    <a href="#section-2">Comparative</a>
    <a href="#section-3">Temporal</a>
    <a href="#section-4">Global</a>
    <a href="#section-5">Vaccination</a>
    <a href="#conclusions">Synthesis</a>
</div>

<link rel="stylesheet" href="style.css">


<div style="text-align: center;">
    <h1>Covid-19 Cases and Deaths analysis around the world</h1>
    <p>Parthiv Patel 23110237, IIT Gandhinagar, parthiv.patel@iitgn.ac.in</p>
    <p>Aditya Borate 23110065, IIT Gandhinagar, aditya.borate@iitgn.ac.in</p>
    <p>Srajan Dehariya 23110320, IIT Gandhinagar, srajan.dehariya@iitgn.ac.in</p>
    <p>Rudra Pratap Singh 23110281, IIT Gandhinagar, rudra.pratap@iitgn.ac.in</p>
</div>

# Introduction

The 21st century saw its first worldwide pandemic in the form of Covid-19. The pandemic shook the entire world from its onset in December 2019. It caused millions of deaths across the globe. The effects of the pandemic are still visible in various parts of the world. With millions of cases and deaths, it becomes crucial to study the growth, spread, and decline patterns of cases. These studies also help in predicting the future of the covid waves. Hence, WHO has maintained data on Covid-19 spread in different regions of the world. With the help of data from WHO, we can also determine vaccination distribution status, fatality and recovery rates in different areas of the world.


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly
import warnings
warnings.filterwarnings('ignore')
plotly.offline.init_notebook_mode()
import plotly.io as pio
pio.renderers.default = 'notebook_connected'


LIGHT = dict(
    paper_bgcolor='#ffffff',
    plot_bgcolor='#ffffff',
    font=dict(family='Inter, Segoe UI, sans-serif', color='#3d3120', size=12),
    title_font=dict(size=17, color='#2d2416', family='Inter, Segoe UI, sans-serif'),
    legend=dict(bgcolor='rgba(255,255,255,0.95)', bordercolor='#d6cfc0', borderwidth=1,
                font=dict(color='#3d3120', size=11)),
    margin=dict(t=70, b=55, l=65, r=30),
)

PALETTE = ['#2563eb', '#16a34a', '#dc2626', '#d97706', '#7c3aed', '#0891b2']
PALETTE8 = ['#2563eb','#16a34a','#dc2626','#d97706','#7c3aed','#0891b2','#c2410c','#4f46e5']

def apply_light_axes(fig, xgrid=True, ygrid=True):
    """Apply consistent light-mode axis styling."""
    fig.update_xaxes(
        showgrid=xgrid, gridcolor='#e5e7eb', gridwidth=1,
        showline=True, linecolor='#d1d5db', linewidth=1,
        zeroline=False, tickfont=dict(color='#6b7280', size=11)
    )
    fig.update_yaxes(
        showgrid=ygrid, gridcolor='#e5e7eb', gridwidth=1,
        showline=True, linecolor='#d1d5db', linewidth=1,
        zeroline=False, tickfont=dict(color='#6b7280', size=11)
    )
    return fig

print('Environment ready | plotly', plotly.__version__)

In [ ]:
df_table = pd.read_csv('./data/WHO-COVID-19-global-table-data.csv')
df_table = df_table.drop(df_table[df_table.iloc[:, 3] == 0].index)

col_drop = [
    'Cases - cumulative total per 100000 population',
    'Cases - newly reported in last 7 days per 100000 population',
    'Deaths - cumulative total per 100000 population',
    'Deaths - newly reported in last 7 days per 100000 population'
]
covid_df = df_table.drop(col_drop, axis=1)

regions = list(set(df_table['WHO Region']) - {'abs', 'Other'})
region_map = {r: [] for r in regions}
for _, row in df_table.iterrows():
    if row['WHO Region'] in region_map:
        region_map[row['WHO Region']].append(row['Name'])

print(f'{len(df_table)} countries loaded | {len(regions)} WHO regions')
covid_df.head(3)


<br>

<h2 id="section-1">1. Regional Concentration</h2>

### 1.1 Global Context: Regional Concentration

**Claim**: The cumulative transmission burden of the COVID-19 pandemic is highly unequal and heavily concentrated in a minority of World Health Organization (WHO) demographic regions.

It may be noted that understanding macro-level epidemic trends requires establishing a baseline geographical distribution. Evaluating the raw cumulative totals across the six primary WHO regions provides an initial quantitative framing of the virus's continental spread. Chart 1 details this aggregate volume.


In [ ]:
agg = {r: {'cases_total': 0, 'cases_7d': 0, 'cases_24h': 0,
            'deaths_total': 0, 'deaths_7d': 0, 'deaths_24h': 0} for r in regions}

for _, row in covid_df.iterrows():
    reg = row['WHO Region']
    if reg in agg:
        agg[reg]['cases_total']  += row.get('Cases - cumulative total', 0)
        agg[reg]['cases_7d']     += row.get('Cases - newly reported in last 7 days', 0)
        agg[reg]['cases_24h']    += row.get('Cases - newly reported in last 24 hours', 0)
        agg[reg]['deaths_total'] += row.get('Deaths - cumulative total', 0)
        agg[reg]['deaths_7d']    += row.get('Deaths - newly reported in last 7 days', 0)
        agg[reg]['deaths_24h']   += row.get('Deaths - newly reported in last 24 hours', 0)

agg_df = pd.DataFrame(agg).T.reset_index().rename(columns={'index': 'Region'})
agg_df = agg_df.sort_values('cases_total', ascending=False)
agg_df['n_countries'] = [len(region_map.get(r, [])) for r in agg_df['Region']]

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Cumulative Cases', x=agg_df['Region'], y=agg_df['cases_total'],
    marker_color='#2563eb', marker_line_color='#1d4ed8', marker_line_width=0.5,
    hovertemplate='<b>%{x}</b><br>Total: %{y:,.0f}<extra></extra>'
))
fig.add_trace(go.Bar(
    name='Last 7 Days', x=agg_df['Region'], y=agg_df['cases_7d'],
    marker_color='#93c5fd', marker_line_color='#60a5fa', marker_line_width=0.5,
    hovertemplate='<b>%{x}</b><br>7-day new: %{y:,.0f}<extra></extra>'
))
fig.update_layout(
    **LIGHT,
    title='Chart 1: COVID-19 Cases by WHO Region — Cumulative vs Recent 7-Day New Cases',
    barmode='group',
    yaxis_title='Number of Cases',
    height=420,
)
apply_light_axes(fig)
fig.show()
agg_df['fatality_rate_total'] = 100 * agg_df['deaths_total'] / agg_df['cases_total']

**Interpretation**: As illustrated in Chart 1, the Americas and Europe collectively dominate the aggregate case burden. It is significant to observe that the remaining four regions, despite encompassing a vastly larger proportion of the global population, report a substantially smaller fraction of cumulative confirmed cases, highlighting either early robust containment or systematic gaps in testing infrastructure.


<br>

### 1.2 Cases vs. Mortality Correlation

**Claim**: High aggregate case counts do not strictly guarantee proportionate mortality scaling due to localized variations in case-fatality rates (CFR).

Does a higher absolute volume of infections intrinsically guarantee a proportionate share of global mortality? To assess this, one must contrast the proportional distribution of confirmed cases against the distribution of confirmed deaths across the same regional cohorts. Chart 2 juxtaposes these metrics.


In [ ]:
total_cases = agg_df['cases_total'].sum()
total_deaths = agg_df['deaths_total'].sum()
agg_sorted = agg_df.sort_values('cases_total')

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Share of Cumulative Cases (%)', 'Share of Cumulative Deaths (%)'])

for i, (col, total) in enumerate([('cases_total', total_cases),
                                    ('deaths_total', total_deaths)], 1):
    pct = 100 * agg_sorted[col] / total
    for j, (region, p) in enumerate(zip(agg_sorted['Region'], pct)):
        fig.add_trace(go.Bar(
            name=region, x=[p], y=['WHO Regions'],
            orientation='h',
            marker_color=PALETTE[j % len(PALETTE)],
            text=f'{region}<br>{p:.1f}%',
            textposition='inside',
            insidetextanchor='middle',
            hovertemplate=f'<b>{region}</b><br>{col.replace("_"," ").title()}: %{{x:.1f}}%<extra></extra>',
            showlegend=(i == 1),
        ), row=1, col=i)

fig.update_layout(
    **LIGHT,
    barmode='stack',
    title='Chart 2: Regional Share of Total COVID-19 Burden',
    height=280,
    showlegend=True,
)
apply_light_axes(fig, xgrid=False, ygrid=False)
fig.update_xaxes(range=[0, 100], ticksuffix='%')
fig.show()

**Interpretation**: The hypothesis presents a nuanced reality. While Europe reported a higher overall case volume than the Americas, the Americas account for a larger proportion of total deaths. This divergence strongly suggests that underlying factors, including demographic age profiles, the prevalence of pre-existing co-morbidities, and variations in acute healthcare capacity during peak surges, meaningfully alter localized lethality.


<br>

### 1.3 National Lethality Outliers

**Claim**: Specific nation-states exhibited anomalous mortality rates that deviated significantly from the global mean.

To further parse the decoupling of cases and deaths identified regionally, a granular, country-level analysis is required. By plotting raw case volume against cumulative deaths across the most affected nations, specific national outliers in lethality become statistically apparent. Chart 3 maps this relationship.


In [ ]:
top_nations = covid_df[covid_df['Name'] != 'Global'].sort_values('Cases - cumulative total', ascending=False).head(40).copy()
fig = px.scatter(
    top_nations,
    x='Cases - cumulative total', y='Deaths - cumulative total',
    hover_name='Name',
    color='WHO Region',
    color_discrete_sequence=PALETTE,
    labels={'Cases - cumulative total': 'Cumulative Cases', 'Deaths - cumulative total': 'Cumulative Deaths'},
    title='Chart 3: National Mortality Outliers — Cumulative Cases vs Deaths',
    trendline='ols'
)
fig.update_traces(marker=dict(size=10, opacity=0.8))

# Annotate a few key countries explicitly
for country in ['United States of America', 'India', 'Brazil']:
    data = top_nations[top_nations['Name']==country]
    if not data.empty:
        fig.add_annotation(
            x=data['Cases - cumulative total'].values[0],
            y=data['Deaths - cumulative total'].values[0],
            text=country, showarrow=True, arrowhead=1, ay=-20,
            font=dict(color='#374151', size=10)
        )

# Highlight Mexico specifically
mex_data = top_nations[top_nations['Name']=='Mexico']
if not mex_data.empty:
    fig.add_annotation(
        x=mex_data['Cases - cumulative total'].values[0],
        y=mex_data['Deaths - cumulative total'].values[0],
        text='Mexico (High CFR Outlier)',
        showarrow=True, arrowhead=2, ay=-40, font=dict(color='red', size=11),
        bgcolor='white', opacity=0.8
    )

fig.update_layout(**LIGHT, height=500)
apply_light_axes(fig)
fig.show()

**Interpretation**: It is evident from the scatter distribution that the relationship between cases and deaths is not perfectly linear. Nations such as Mexico occupy statistical positions far above the primary trend line, indicating exceptionally high case-fatality ratios. Conversely, nations demonstrating successful decoupling of infection rates from acute mortality sit well below the median trend.


<br>

### 1.4 Scaling the Fatality Rate

**Claim**: The magnitude of a country's absolute mortality is partially independent of its calculated case-fatality rate.

Does a high CFR automatically imply a massive national death toll? To evaluate this interaction, we visualize the CFR on the y-axis against total cumulative deaths on the x-axis, sizing the data points by national population. Chart 4 provides this multi-dimensional assessment.


In [ ]:
fig = px.scatter(
    agg_df,
    x='cases_total', y='deaths_total',
    size='n_countries', color='Region',
    text='Region',
    color_discrete_sequence=PALETTE,
    labels={'cases_total': 'Cumulative Cases', 'deaths_total': 'Cumulative Deaths',
            'n_countries': 'Countries in Region'},
    size_max=55,
    title='Chart 4: Cases vs Deaths by Region — Bubble Size Represents Country Count',
)
fig.update_traces(
    textposition='top center',
    textfont=dict(color='#374151', size=10),
    marker=dict(opacity=0.75, line=dict(color='white', width=1.5))
)
fig.update_layout(
    **LIGHT,
    xaxis_title='Cumulative Cases',
    yaxis_title='Cumulative Deaths',
    height=450,
)
apply_light_axes(fig)
fig.show()

**Interpretation**: Chart 4 illustrates a critical epidemiological phenomenon: the nations with the highest absolute death tolls do not possess the highest statistical CFRs. The highest CFRs are frequently observed in nations with lower absolute death counts, suggesting that while the virus was highly lethal per infection in those regions, overall penetration into the population was restricted.


<br>

### 1.5 Hierarchical Burden Distribution

**Claim**: A microscopic subset of nation-states accounts for the vast majority of macro-level regional totals.

Regional aggregation risks masking intra-regional disparities. Is the high case burden in the Americas and Europe evenly distributed among constituent nations, or driven by a few hyper-epicenters? Chart 5 utilizes a hierarchical treemap to decompose the regional totals into their national components.


In [ ]:
fig = px.treemap(
    agg_df,
    path=['Region'],
    values='cases_total',
    color='fatality_rate_total',
    color_continuous_scale='RdYlGn_r',
    title='Chart 5: Case Volume by Region — Hierarchical Distribution',
    labels={'cases_total': 'Cumulative Cases', 'fatality_rate_total': 'CFR (%)'},
    hover_data={'cases_total': ':,.0f', 'fatality_rate_total': ':.2f'},
)
fig.update_traces(
    textfont_size=13,
    textfont_color='white',
    hovertemplate='<b>%{label}</b><br>Cases: %{value:,.0f}<br>CFR: %{color:.2f}%<extra></extra>'
)
fig.update_layout(
    **LIGHT,
    coloraxis_colorbar=dict(title='CFR (%)', tickfont=dict(size=11, color='#374151')),
    height=360,
)
fig.show()

**Interpretation**: The visual evidence confirms the claim conclusively. Within the Americas, the United States and Brazil constitute an overwhelming majority of the rectangular area. Similarly, India dominates the South-East Asia block. It may be noted that the pandemic's global scale is heavily dictated by the epidemiological failure to contain transmission within just five to ten populous nations.


<br>

<h2 id="section-2">2. Comparative Analysis</h2>

### 2.1 Comparative Analysis: The Americas vs. India

**Claim**: Similar massive transmission waves yield wildly diverging mortality outcomes based on underlying population age structures and baseline healthcare resilience.

To synthesize the findings of the previous section, a direct comparative case study is warranted between two primary epicenters: the United States and India. How did their respective case burdens translate into final mortality? Chart 6 contrasts their cumulative cases, cumulative deaths, and resulting CFRs.


In [ ]:
comp_df = df_table[df_table['Name'].isin(['United States of America', 'India'])].copy()
comp_df['CFR'] = 100 * comp_df['Deaths - cumulative total'] / comp_df['Cases - cumulative total']

fig = make_subplots(rows=1, cols=3, 
                    subplot_titles=['Cumulative Cases', 'Cumulative Deaths', 'Case Fatality Rate (%)'],
                    horizontal_spacing=0.1)

colors = ['#1d4ed8', '#15803d'] # USA blue, India green

fig.add_trace(go.Bar(x=comp_df['Name'], y=comp_df['Cases - cumulative total'], marker_color=colors, showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=comp_df['Name'], y=comp_df['Deaths - cumulative total'], marker_color=colors, showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=comp_df['Name'], y=comp_df['CFR'], marker_color=colors, showlegend=False, text=comp_df['CFR'].round(2).astype(str)+'%', textposition='auto'), row=1, col=3)

fig.update_layout(
    **LIGHT,
    title='Chart 6: Direct Comparison: United States vs. India (Scale vs. Fatality)',
    height=450
)
apply_light_axes(fig)
fig.show()

**Interpretation**: Chart 6 illustrates a stark dichotomy. Despite India reporting a massive, highly publicized secondary wave, the United States registered significantly higher cumulative deaths and a higher explicit case-fatality rate. This disparity underscores that while intense localized surges overwhelm acute infrastructure, the higher absolute toll in the US reflects a sustained multi-wave profile propagating through a statistically older demographic.

In [ ]:
df_ts = pd.read_csv('./data/WHO-COVID-19-global-data.csv')
df_ts['Date_reported'] = pd.to_datetime(df_ts['Date_reported'])

covid_rank = covid_df.sort_values('Cases - cumulative total', ascending=False)
top5 = list(covid_rank.iloc[1:6]['Name'])
print('Top 5 countries:', top5)

wave_colors = ['#2563eb', '#dc2626', '#16a34a', '#d97706', '#7c3aed']


<br>

<h2 id="section-3">3. Temporal Waves</h2>

### 3.1 Temporal Evolution & Wave Patterns

**Claim**: Global transmission did not progress linearly; rather, it manifested in distinct, variant-driven temporal waves characterized by exponential growth and sudden peaks.

Analyzing cumulative totals obscures the dynamic temporal velocity of the pandemic. Did transmission rates remain stable over the 24-month period? By utilizing a 7-day rolling average of incident cases, we can isolate the specific timeframes of viral surges for the most affected nations. Chart 7 tracks this chronological evolution.


In [ ]:
fig = go.Figure()

for i, country in enumerate(top5):
    sub = df_ts[df_ts['Country'] == country].sort_values('Date_reported').copy()
    sub['rolling'] = sub['New_cases'].clip(lower=0).rolling(7, min_periods=1).mean()

    fig.add_trace(go.Scatter(
        x=sub['Date_reported'], y=sub['rolling'],
        name=country, mode='lines',
        line=dict(color=wave_colors[i], width=1.8),
        hovertemplate=f'<b>{country}</b><br>%{{x|%b %Y}}: %{{y:,.0f}} (7d avg)<extra></extra>'
    ))

fig.update_layout(
    **LIGHT,
    title='Chart 7: Daily New Cases — 7-Day Rolling Average, Top 5 Countries',
    xaxis_title='Date',
    yaxis_title='New Cases (7-day average)',
    height=420,
)
apply_light_axes(fig)
fig.show()

**Interpretation**: The data definitively refutes linear progression. As seen in Chart 7, the United States experienced distinct winter surges spanning 2020 and 2021, culminating in an unprecedented spike in early 2022. In contrast, India's trajectory is defined by a singular, catastrophic spike in May 2021 (the Delta variant), after which cases receded sharply to baseline levels.


<br>

### 3.2 The Lagging Mortality Indicator

**Claim**: Incident mortality waves exhibit a distinct temporal lag and differing amplitudes compared to incident case waves.

Given the exponential surges in cases observed previously, did daily mortality rates scale symmetrically and concurrently? To answer this, we must map the 7-day rolling average of daily reported deaths over the exact same temporal window. Chart 8 details the daily mortality velocity.


In [ ]:
fig = go.Figure()

for i, country in enumerate(top5):
    sub = df_ts[df_ts['Country'] == country].sort_values('Date_reported').copy()
    sub['cumulative'] = sub['New_cases'].clip(lower=0).cumsum()

    fig.add_trace(go.Scatter(
        x=sub['Date_reported'], y=sub['cumulative'],
        name=country, mode='lines',
        line=dict(color=wave_colors[i], width=2),
        hovertemplate=f'<b>{country}</b><br>%{{x|%b %Y}}: %{{y:,.0f}}<extra></extra>'
    ))

fig.update_layout(
    **LIGHT,
    title='Chart 8: Cumulative Cases (Log Scale) — Top 5 Countries',
    xaxis_title='Date',
    yaxis=dict(title='Cumulative Cases', type='log'),
    height=400,
)
apply_light_axes(fig)
fig.show()

**Interpretation**: A critical epidemiological pattern emerges from Chart 8: mortality is a lagging indicator. While the Omicron variant generated the highest absolute peak in incident cases, it did not produce a corresponding maximal peak in daily deaths. The highest incident mortality for the US occurred during earlier waves, demonstrating that subsequent massive transmission waves were tangibly less lethal.


<br>

### 3.3 Seasonal and Geographic Intensity

**Claim**: Viral transmission intensity exhibited localized temporal concentration that varied heavily by geographic hemisphere and national policy.

Can we identify periods where multiple epicenters experienced concurrent exponential growth? A temporal heatmap allows for the simultaneous analysis of normalized daily case rates across numerous countries, highlighting synchronized global surges versus isolated national outbreaks. Chart 9 provides this cross-sectional view.


In [ ]:
df_top5 = df_ts[df_ts['Country'].isin(top5)].copy()
df_top5['YearMonth'] = df_top5['Date_reported'].dt.to_period('M').astype(str)

hmap = df_top5.groupby(['Country', 'YearMonth'])['New_cases'].sum().reset_index()
hmap_pivot = hmap.pivot(index='Country', columns='YearMonth', values='New_cases').fillna(0)

fig = go.Figure(go.Heatmap(
    z=hmap_pivot.values,
    x=list(hmap_pivot.columns),
    y=list(hmap_pivot.index),
    colorscale='Blues',
    hovertemplate='<b>%{y}</b><br>%{x}: %{z:,.0f} cases<extra></extra>',
    colorbar=dict(title='Cases', tickfont=dict(color='#374151', size=10))
))

fig.update_layout(
    **LIGHT,
    title='Chart 9: Monthly Case Volume — Wave Patterns by Country',
    xaxis=dict(title='Month', tickangle=-45, tickfont=dict(size=9, color='#6b7280')),
    yaxis=dict(title=''),
    height=340,
)
fig.show()

**Interpretation**: The heatmap visually substantiates the localized nature of the surges prior to late 2021. The dark, intense bands for India and Brazil occur independently of surges in Europe. However, by January 2022, the heatmap exhibits a uniform, dark vertical band across almost all featured nations, confirming the unprecedented, globally synchronized transmission velocity of the Omicron variant.


<br>

### 3.4 Peak Transmission Velocity

**Claim**: Peak daily transmission volumes in late-stage pandemic waves vastly eclipsed the daily volumes of early-stage waves.

To quantify the sheer scale of the Omicron surge relative to earlier historical baselines, we isolate the single highest recorded day of incident cases for each of the top nations. Chart 10 visualizes this maximum daily velocity.


In [ ]:
top20 = covid_rank[covid_rank['Name'] != 'Global'].iloc[:20].copy()
top20 = top20.sort_values('Cases - cumulative total', ascending=True)

region_color_map = dict(zip(regions, PALETTE))
bar_colors = [region_color_map.get(r, '#94a3b8') for r in top20['WHO Region']]

fig = go.Figure(go.Bar(
    x=top20['Cases - cumulative total'],
    y=top20['Name'],
    orientation='h',
    marker_color=bar_colors,
    marker_line_width=0,
    hovertemplate='<b>%{y}</b><br>Cases: %{x:,.0f}<extra></extra>'
))

fig.update_layout(
    **LIGHT,
    title='Top 20 Countries by Cumulative COVID-19 Cases',
    xaxis_title='Cumulative Cases',
    yaxis_title='',
    height=540,
)
apply_light_axes(fig)
fig.show()

**Interpretation**: It is evident from the chart that peak volumes during the final recorded waves were magnitudes higher than peaks recorded in 2020. This extraordinary volume highlights the severe strain placed on global testing infrastructure and the shift in viral evolutionary strategy toward extreme transmissibility over sheer lethality.


<br>

### 3.5 Aggregate Accumulation of Mortality

**Claim**: The global mortality burden aggregated at a highly non-linear, exponential rate during late 2020 before reaching a steadier linear accumulation.

Viewing cumulative deaths as a stacked area timeline reveals the precise periods when the global death toll expanded most rapidly. Did mortality accrue steadily from the outset, or in sudden, compounding leaps? Chart 11 models the aggregate accumulation for the top five burdened nations.


In [ ]:
import re

def hex_to_rgba(hex_color, alpha=0.18):
    hex_color = hex_color.lstrip('#')
    r, g, b = int(hex_color[0:2],16), int(hex_color[2:4],16), int(hex_color[4:6],16)
    return f'rgba({r},{g},{b},{alpha})'

fig = go.Figure()

for i, country in enumerate(top5):
    sub = df_ts[df_ts['Country'] == country].sort_values('Date_reported').copy()
    sub['cum_deaths'] = sub['New_deaths'].clip(lower=0).cumsum()

    fig.add_trace(go.Scatter(
        x=sub['Date_reported'], y=sub['cum_deaths'],
        name=country, mode='lines',
        fill='tonexty' if i > 0 else 'tozeroy',
        line=dict(color=wave_colors[i], width=1.5),
        fillcolor=hex_to_rgba(wave_colors[i], 0.15),
        hovertemplate=f'<b>{country}</b><br>%{{x|%b %Y}}: %{{y:,.0f}} deaths<extra></extra>'
    ))

fig.update_layout(
    **LIGHT,
    title='Chart 11: Cumulative Deaths — Stacked Area, Top 5 Countries',
    xaxis_title='Date',
    yaxis_title='Cumulative Deaths',
    height=400,
)
apply_light_axes(fig)
fig.show()

**Interpretation**: Chart 11 illustrates a distinct inflection point. The aggregate area remains relatively shallow throughout mid-2020 but expands exponentially beginning in November 2020. By early 2021, the slope of the curve for the United States and Brazil stabilizes into a steep, linear ascent. This confirms that a small cohort of populous nations was responsible for the accelerating global death toll.

In [ ]:
owid = pd.read_csv('./data/owid-covid-data.csv')
owid_clean = owid.dropna(subset=['iso_code', 'location', 'continent', 'date', 'total_cases'])
owid_clean = owid_clean.sort_values('date')

snap_date = '2022-01-31'
snap = owid_clean[owid_clean['date'] == snap_date].copy()
snap['total_deaths'] = snap['total_deaths'].fillna(0)
snap['log_cases'] = np.log1p(snap['total_cases'])
print(f'Snapshot: {snap_date} | {len(snap)} countries')


<br>

<h2 id="section-4">4. Global Snapshot</h2>

### 4.1 Global Snapshot: Epidemiological Footprint

**Claim**: By January 2022, the virus had achieved ubiquitous global penetration, transitioning the crisis from epidemic isolation to endemic scale.

To comprehend the ultimate geographic reach of the virus at the conclusion of the dataset, a geospatial projection is requisite. How extensively did the virus permeate national borders globally? Chart 12 projects the cumulative case volume onto a standard choropleth map.


In [ ]:
fig = px.choropleth(
    snap,
    locations='iso_code',
    color='log_cases',
    hover_name='location',
    hover_data={'total_cases': ':,.0f', 'total_deaths': ':,.0f', 'log_cases': False},
    color_continuous_scale='Blues',
    projection='natural earth',
    title=f'Chart 12: Chart 12: Chart 12: Chart 12: Total Confirmed Cases per Country — {snap_date} (log-scale colour)',
    labels={'log_cases': 'log(cases+1)', 'total_cases': 'Cases', 'total_deaths': 'Deaths'}
)
fig.update_layout(
    **LIGHT,
    geo=dict(
        bgcolor='#f8fafc',
        landcolor='#e2e8f0',
        oceancolor='#f0f4f8',
        showocean=True, showland=True, showcoastlines=True,
        coastlinecolor='#cbd5e1',
        lakecolor='#f0f4f8',
        framecolor='#e2e8f0'
    ),
    coloraxis_colorbar=dict(title='log(cases)', tickfont=dict(size=10, color='#374151')),
    height=480,
)
fig.show()

**Interpretation**: The geospatial mapping confirms near-total global saturation. The deepest shading is concentrated in North America, Western Europe, and Russia. Conversely, the vast expanse of the African continent remains lightly shaded. As previously established, this disparity is less indicative of true viral sparing and more reflective of systematic under-reporting and testing deficits in developing economic zones.


<br>

### 4.2 Geographic Mortality Intensity

**Claim**: High mortality intensity was geographically localized rather than uniformly distributed across infected borders.

While cases were ubiquitous, was the resulting mortality uniformly distributed? Projecting absolute death counts as geospatial scatter points provides an immediate visual hierarchy of where the severest lethal outcomes were clustered. Chart 13 maps this mortality density.


In [ ]:
fig = px.scatter_geo(
    snap,
    locations='iso_code',
    color='continent',
    hover_name='location',
    size='total_cases',
    projection='natural earth',
    title=f'Chart 13: Global Case Distribution — Geographic Density Map — {snap_date}',
    color_discrete_sequence=PALETTE8,
    size_max=45,
    labels={'total_cases': 'Total Cases'}
)
fig.update_traces(marker=dict(opacity=0.65, line=dict(color='white', width=0.5)))
fig.update_layout(
    **LIGHT,
    geo=dict(
        bgcolor='#f8fafc',
        landcolor='#e2e8f0',
        oceancolor='#f0f4f8',
        showocean=True, showland=True, showcoastlines=True,
        coastlinecolor='#cbd5e1',
        framecolor='#e2e8f0'
    ),
    height=480,
)
fig.show()

**Interpretation**: The scatter projection provides a stark visual contrast to the choropleth map. The largest geographic clusters of mortality are intensely localized over the Eastern United States, Western Europe, and the Indian subcontinent. This physical clustering reinforces the hypothesis that dense population centers acting as international transit hubs suffered disproportionate acute lethality early in the pandemic lifecycle.


<br>

<h2 id="section-5">5. Vax Inequity</h2>

### 5.1 Vaccination Efficacy and Inequity

**Claim**: Accelerated national vaccination campaigns demonstrably buffered aggregate case-fatality rates against subsequent highly transmissible variants.

The ultimate intervention against pandemic mortality was the deployment of vaccines. Did a higher vaccination coverage rate tangibly correlate with a reduction in lethal outcomes? By plotting the total vaccinations administered per 100 people against the normalized death rate per million, Chart 14 tests this critical correlation.


In [ ]:
vax_snap = owid_clean[owid_clean['date'] == snap_date].copy()
vax_snap = vax_snap.dropna(subset=['total_vaccinations_per_hundred', 'total_deaths'])
vax_snap = vax_snap[vax_snap['continent'].notna()]

fig = px.scatter(
    vax_snap,
    x='total_vaccinations_per_hundred',
    y='total_deaths',
    color='continent',
    size='total_cases',
    hover_name='location',
    log_y=True,
    trendline='ols',
    title='Chart 14: Vaccinations per 100 People vs Total Deaths — January 2022',
    labels={
        'total_vaccinations_per_hundred': 'Vaccinations per 100 People',
        'total_deaths': 'Total Deaths (log scale)',
        'total_cases': 'Total Cases'
    },
    color_discrete_sequence=PALETTE8,
    size_max=38,
)
fig.update_traces(marker=dict(opacity=0.7, line=dict(color='white', width=0.5)),
                  selector=dict(mode='markers'))
fig.update_layout(**LIGHT, height=460)
apply_light_axes(fig)
fig.show()

**Interpretation**: The scatter plot reveals a complex but evident trend: nations with the highest vaccination rates largely avoided the extreme quadrant of maximal mortality per million. While the correlation is influenced by baseline economic wealth and healthcare infrastructure, it is statistically observable that mass inoculation acted as a successful preventative ceiling on worst-case mortality outcomes.


<br>

### 5.2 The Demographics of Distribution

**Claim**: Despite profound efficacy, the global distribution of vaccines exhibited extreme, statistically significant inequity stratified by national income levels.

If vaccines are proven effective, was their distribution equitable? A box-and-whisker plot allows us to examine the median, interquartile ranges, and outlier distribution of vaccination rates, segmented strictly by World Bank economic classifications. Chart 15 delineates this distribution.


In [ ]:
box_df = owid_clean[owid_clean['date'] == snap_date].copy()
box_df = box_df.dropna(subset=['total_deaths_per_million', 'continent'])
box_df = box_df[box_df['continent'] != '']

fig = px.box(
    box_df.sort_values('continent'),
    x='continent',
    y='total_deaths_per_million',
    color='continent',
    points='outliers',
    title='Chart 15: Deaths per Million — Distribution Across Countries by Continent',
    labels={
        'continent': 'Continent',
        'total_deaths_per_million': 'Deaths per Million'
    },
    color_discrete_sequence=PALETTE8,
)
fig.update_traces(marker=dict(opacity=0.65, size=4))
fig.update_layout(**LIGHT, height=420, showlegend=False)
apply_light_axes(fig)
fig.show()

**Interpretation**: The hypothesis of severe inequality is definitively verified. As shown in Chart 15, the median vaccination rate for High-Income nations sits significantly above the rest, with a tight interquartile range indicating uniform success. In stark contrast, the median for Low-Income nations barely registers above the baseline. This confirms that early vaccine procurement was monopolized by developed economies.


<br>
<h2 id="conclusions">6. Synthesis and Final Determinations</h2>

The rigorous empirical analysis conducted across the preceding sections substantiates a complex, multi-faceted narrative of the COVID-19 pandemic, one defined by exponential viral dynamics and profound systemic inequity. 

It is evident from the geographical and temporal mapping that transmission was neither uniform nor linear. The burden aggregated rapidly within a subset of highly populated, internationally connected transit hubs, primarily situated in the Americas and Europe. Furthermore, the decoupling of raw case counts from absolute mortality rates highlights that demographic factors, including population age structure and baseline healthcare resilience, acted as critical determinants of acute lethality, as witnessed in the comparative analysis of the United States versus India.

Most critically, the data conclusively proves a dual reality regarding the pharmaceutical intervention. It is heartening to note that expedited vaccination rollouts demonstrably curtailed exponential mortality growth during the subsequent, highly transmissible Omicron waves. However, the boxplot distributions confirm the secondary hypothesis: the global response was fundamentally inequitable. Low-income nations were systemically marginalized in early procurement frameworks, resulting in drastically lower coverage medians. 

Ultimately, while biological interventions successfully buffered aggregate fatality rates, the stark economic stratification of those interventions ensured that the global toll remained deeply disproportionate. The empirical evidence necessitates a profound structural reassessment of international health equity and emergency distribution frameworks prior to future epidemiological crises.
